# VQE 教程：顶层接口 + 手动 Parameter-Shift 梯度下降

本教程分两部分：
1. 顶层：直接使用 `VQERunner.run_model` 跑 VQE
2. 拆解：手动实现 `parameter-shift` 梯度 + 梯度下降更新

默认用 `Simulator`，便于稳定复现与快速实验。

In [1]:
import numpy as np

from quantum_hw import QuantumHardwareClient, VQERunner, QuantumCircuit
from quantum_hw.api import Backend
from quantum_hw.algorithms.vqe import (
    build_ising_hamiltonian,
    build_hardware_efficient_ansatz,
    _build_hardware_efficient_ansatz_symbolic,
    _evaluate_energy_with_backend,
    _parameter_shift_gradient,
)

def section(title: str):
    print("\n" + "=" * 18, title, "=" * 18)

## 1) 问题设置（Ising 哈密顿量）

In [2]:
section("setup")
num_qubits = 2
layers = 1
shots = 2048
max_iters = 30
learning_rate = 0.1
shift = np.pi / 2
seed = 42




================== setup ==================


## 2) 顶层调用：`VQERunner.run_model`

直接把优化细节交给框架。

In [3]:
section("high-level VQE")
client = QuantumHardwareClient()
vqe = VQERunner(
    client=client,
    layers=layers,
    shots=shots,
    max_iters=max_iters,
    learning_rate=learning_rate,
    shift=shift,
    zne=False,
    readout_mitigation=False,
    seed=seed,
)

result_top = vqe.run_model(
    name="tutorial_vqe_top",
    num_qubits=num_qubits,
    model="ising",
    model_params={"j": 1.0, "h": 1.0},
    prefer_chips="Simulator",
)

print("best_energy:", result_top.best_energy)
print("best_params (first 6):", result_top.best_params[:6])
print("energy_history:", result_top.energy_history)


================== high-level VQE ==================
[vqe] prepare run: name=tutorial_vqe_top num_qubits=2 model=ising layers=1 shots=2048 max_iters=30
[vqe] candidate chips: ['Simulator']
[vqe] running on chip: Simulator
[vqe] start optimization: iters=30 layers=1 params=8 shots=2048 shift=1.5707963267948966 gradient=parameter-shift
[vqe] iter 0 start
[vqe] iter 0 energy=0.791016 grad_norm=1.898899
[vqe] iter 0 new best energy=0.791016
[vqe] iter 1 start
[vqe] iter 1 energy=0.335938 grad_norm=2.056771
[vqe] iter 1 new best energy=0.335938
[vqe] iter 2 start
[vqe] iter 2 energy=-0.138672 grad_norm=1.957343
[vqe] iter 2 new best energy=-0.138672
[vqe] iter 3 start
[vqe] iter 3 energy=-0.520508 grad_norm=1.649440
[vqe] iter 3 new best energy=-0.520508
[vqe] iter 4 start
[vqe] iter 4 energy=-0.864258 grad_norm=1.147589
[vqe] iter 4 new best energy=-0.864258
[vqe] iter 5 start
[vqe] iter 5 energy=-1.032227 grad_norm=0.651419
[vqe] iter 5 new best energy=-1.032227
[vqe] iter 6 start
[vqe] 

## 3) 拆解版：手动写能量评估 + Parameter-Shift 梯度

思路：
- 给定参数 `params` 构造 ansatz 线路
- 用 `client.run_auto(... observables=...)` 得到每个 Pauli 项期望
- 按哈密顿量线性组合得到能量
- 用 parameter-shift 公式算梯度

In [4]:
section("manual energy + gradient")
client_manual = QuantumHardwareClient()

def evaluate_energy(qc: QuantumCircuit, run_name: str, num_qubits: int, backend: Backend, shots: int, observables): # demo code for manual energy evaluation, also see _evaluate_energy_with_backend()
    result = client_manual._run_with_backend(
    qc,
    run_name,
    num_qubits,
    backend=backend,
    chip_name="Simulator",
    shots=shots,
    observables=observables,
    transpile=False,
    )
    exp_map = result.observable_values
    energy = float(sum(coeff * exp_map.get(obs, 0.0) for coeff, obs in hamiltonian))
    return energy, exp_map

def parameter_shift_grad(params: np.ndarray, shift_value: float, run_name: str, num_qubits: int, backend: Backend, shots: int, observables, layers: int): # demo code for manual gradient evaluation using parameter-shift rule, also see _parameter_shift_gradient()
    grads = np.zeros_like(params, dtype=float)
    for i in range(params.size):
        p_plus = params.copy()
        p_minus = params.copy()
        p_plus[i] += shift_value
        p_minus[i] -= shift_value
        qc_plus = build_hardware_efficient_ansatz(num_qubits, p_plus, layers=layers)
        qc_minus = build_hardware_efficient_ansatz(num_qubits, p_minus, layers=layers)
        e_plus, _ = evaluate_energy(qc_plus, f"{run_name}_grad_p{i}", num_qubits, backend, shots, observables)
        e_minus, _ = evaluate_energy(qc_minus, f"{run_name}_grad_m{i}", num_qubits, backend, shots, observables)
        grads[i] = 0.5 * (e_plus - e_minus)
    return grads

# 构造Hamiltonian和ansatz
hamiltonian = build_ising_hamiltonian(num_qubits=num_qubits, j=1.0, h=1.0)
observables = [obs for _, obs in hamiltonian]
print("hamiltonian terms:")
for coeff, obs in hamiltonian:
    print(f"  {coeff:+.3f} * {obs}")

num_params = 2 * num_qubits * (layers + 1)
rng = np.random.default_rng(seed)
params0 = rng.uniform(0.0, 2.0 * np.pi, size=num_params)
qc = build_hardware_efficient_ansatz(num_qubits, params0, layers=layers)
backend = Backend("Simulator")
# 调用demo函数
e0, exp0 = evaluate_energy(qc, "tutorial_vqe_manual_init", num_qubits, backend, shots, observables)
g0 = parameter_shift_grad(params0, shift, "tutorial_vqe", num_qubits, backend, shots, observables, layers)
print("initial energy:", e0)
print("initial grad norm:", float(np.linalg.norm(g0)))
print("initial expectations:", exp0)



================== manual energy + gradient ==================
hamiltonian terms:
  -1.000 * Z0 Z1
  -1.000 * X0
  -1.000 * X1
initial energy: 0.791015625
initial grad norm: 1.898899163520376
initial expectations: {'Z0 Z1': -0.2861328125, 'X0': -0.23828125, 'X1': -0.2666015625}


## 4) 手动梯度下降循环

这里用最朴素的 GD 更新：
$$\theta \leftarrow \theta - \eta \nabla E(\theta)$$

In [5]:
section("manual gradient descent")
params = params0.copy()
energy_history_manual = []

for it in range(max_iters):
    qc = build_hardware_efficient_ansatz(num_qubits, params, layers=layers)
    energy, _ = evaluate_energy(qc, f"tutorial_vqe_manual_iter{it}", num_qubits, backend, shots, observables)
    grads = parameter_shift_grad(params, shift_value=shift, run_name=f"tutorial_vqe_manual_iter{it}", num_qubits=num_qubits, backend=backend, shots=shots, observables=observables, layers=layers)
    grad_norm = float(np.linalg.norm(grads))

    energy_history_manual.append(float(energy))
    print(f"iter={it:02d} energy={energy:.6f} grad_norm={grad_norm:.6f}")

    params = params - learning_rate * grads

energy_final, exp_final = evaluate_energy(qc, run_name="tutorial_vqe_manual_final", num_qubits=num_qubits, backend=backend, shots=shots, observables=observables)
print("manual final energy:", energy_final)
print("manual final expectations:", exp_final)


================== manual gradient descent ==================
iter=00 energy=0.791016 grad_norm=1.898899
iter=01 energy=0.415039 grad_norm=2.067400
iter=02 energy=-0.023438 grad_norm=2.088364
iter=03 energy=-0.436523 grad_norm=1.986921
iter=04 energy=-0.797852 grad_norm=1.741982
iter=05 energy=-1.078125 grad_norm=1.446582
iter=06 energy=-1.283203 grad_norm=1.198701
iter=07 energy=-1.426758 grad_norm=0.980883
iter=08 energy=-1.523438 grad_norm=0.838749
iter=09 energy=-1.579102 grad_norm=0.753094
iter=10 energy=-1.630859 grad_norm=0.691643
iter=11 energy=-1.686523 grad_norm=0.666976
iter=12 energy=-1.736328 grad_norm=0.645996
iter=13 energy=-1.778320 grad_norm=0.622375
iter=14 energy=-1.806641 grad_norm=0.607959
iter=15 energy=-1.840820 grad_norm=0.593813
iter=16 energy=-1.881836 grad_norm=0.594847
iter=17 energy=-1.908203 grad_norm=0.578688
iter=18 energy=-1.930664 grad_norm=0.558876
iter=19 energy=-1.961914 grad_norm=0.542915
iter=20 energy=-1.992188 grad_norm=0.528811
iter=21 energy=

## 5) 顶层与手动版本对比

In [6]:
section("compare")
print("top best energy:", result_top.best_energy)
print("manual final energy:", energy_final)
print("top history:", result_top.energy_history)
print("manual history:", energy_history_manual)


================== compare ==================
top best energy: -2.2333984375
manual final energy: -2.1474609375
top history: [0.791015625, 0.3359375, -0.138671875, -0.5205078125, -0.8642578125, -1.0322265625, -1.0888671875, -1.0517578125, -0.98046875, -0.9326171875, -0.94921875, -1.0205078125, -1.1416015625, -1.2900390625, -1.447265625, -1.5771484375, -1.708984375, -1.818359375, -1.8818359375, -1.9443359375, -2.01171875, -2.0625, -2.1220703125, -2.16015625, -2.189453125, -2.2216796875, -2.2333984375, -2.2158203125, -2.220703125, -2.1767578125]
manual history: [0.791015625, 0.4150390625, -0.0234375, -0.4365234375, -0.7978515625, -1.078125, -1.283203125, -1.4267578125, -1.5234375, -1.5791015625, -1.630859375, -1.6865234375, -1.736328125, -1.7783203125, -1.806640625, -1.8408203125, -1.8818359375, -1.908203125, -1.9306640625, -1.9619140625, -1.9921875, -2.029296875, -2.0478515625, -2.0615234375, -2.0751953125, -2.0966796875, -2.11328125, -2.1240234375, -2.138671875, -2.1474609375]


# 6) 自动微分
在设计算法时，可使用自动微分

In [6]:
section("auto-grad VQE")
client = QuantumHardwareClient()
vqe = VQERunner(
    client=client,
    layers=layers,
    shots=shots,
    max_iters=max_iters,
    learning_rate=learning_rate,
    shift=shift,
    zne=False,
    readout_mitigation=False,
    seed=seed,
    gradient_method="autograd",
)

result_top = vqe.run_model(
    name="tutorial_vqe_top",
    num_qubits=num_qubits,
    model="ising",
    model_params={"j": 1.0, "h": 1.0},
    prefer_chips="Simulator",
)

print("best_energy:", result_top.best_energy)
print("best_params (first 6):", result_top.best_params[:6])
print("energy_history:", result_top.energy_history)


================== auto-grad VQE ==================
[vqe] prepare run: name=tutorial_vqe_top num_qubits=2 model=ising layers=1 shots=2048 max_iters=30
[vqe] candidate chips: ['Simulator']
[vqe] running on chip: Simulator
[vqe] start optimization: iters=30 layers=1 params=8 shots=2048 shift=1.5707963267948966 gradient=autograd
[vqe] iter 0 start
[vqe] iter 0 energy=0.821374 grad_norm=1.900223
[vqe] iter 0 new best energy=0.821374
[vqe] iter 1 start
[vqe] iter 1 energy=0.362972 grad_norm=2.058591
[vqe] iter 1 new best energy=0.362972
[vqe] iter 2 start
[vqe] iter 2 energy=-0.111528 grad_norm=1.979099
[vqe] iter 2 new best energy=-0.111528
[vqe] iter 3 start
[vqe] iter 3 energy=-0.535738 grad_norm=1.663019
[vqe] iter 3 new best energy=-0.535738
[vqe] iter 4 start
[vqe] iter 4 energy=-0.852744 grad_norm=1.174930
[vqe] iter 4 new best energy=-0.852744
[vqe] iter 5 start
[vqe] iter 5 energy=-1.031044 grad_norm=0.663205
[vqe] iter 5 new best energy=-1.031044
[vqe] iter 6 start
[vqe] iter 6 e

# 7) 真实硬件完整梯度下降

下面演示真实硬件路径的完整流程：
1. 先对参数化 ansatz 预编译一次
2. 每轮只替换参数值做能量评估与 parameter-shift 梯度
3. 用最朴素的 GD 更新参数

In [ ]:
section("hardware full GD loop")

chip_name_hw = "Baihua"
hardware_iters = 6
zne_hw = True
readout_hw = True

backend = Backend(chip_name_hw)
param_names = [f"theta_{i}" for i in range(num_params)]
symbolic_qc = _build_hardware_efficient_ansatz_symbolic(
    num_qubits,
    param_names,
    layers=layers,
 )

# compile once on hardware backend
transpiled_template = client_manual._transpile_with_backend(
    symbolic_qc,
    backend,
    use_gate_compressor=False,
)

params_hw = params0.copy()
energy_history_hw = []

for it in range(hardware_iters):
    qc_it = transpiled_template.deepcopy()
    qc_it.apply_value({name: float(params_hw[i]) for i, name in enumerate(param_names)}, deep=True)

    energy, _ = _evaluate_energy_with_backend(
        client_manual,
        qc_it,
        name=f"tutorial_vqe_hw_iter{it}",
        num_qubits=num_qubits,
        backend=backend,
        chip_name=chip_name_hw,
        shots=shots,
        hamiltonian=hamiltonian,
        zne=zne_hw,
        readout_mitigation=readout_hw,
    )

    grads = _parameter_shift_gradient(
        client_manual,
        params_hw,
        name=f"tutorial_vqe_hw_iter{it}",
        num_qubits=num_qubits,
        backend=backend,
        chip_name=chip_name_hw,
        shots=shots,
        hamiltonian=hamiltonian,
        shift=shift,
        zne=zne_hw,
        readout_mitigation=readout_hw,
        transpiled_template=transpiled_template,
        param_names=param_names,
    )

    grad_norm = float(np.linalg.norm(grads))
    energy_history_hw.append(float(energy))
    print(f"iter={it:02d} energy={energy:.6f} grad_norm={grad_norm:.6f}")

    params_hw = params_hw - learning_rate * grads

qc_final = transpiled_template.deepcopy()
qc_final.apply_value({name: float(params_hw[i]) for i, name in enumerate(param_names)}, deep=True)
energy_final_hw, exp_final_hw = _evaluate_energy_with_backend(
    client_manual,
    qc_final,
    name="tutorial_vqe_hw_final",
    num_qubits=num_qubits,
    backend=backend,
    chip_name=chip_name_hw,
    shots=shots,
    hamiltonian=hamiltonian,
    zne=zne_hw,
    readout_mitigation=readout_hw,
)

print("hardware final energy:", energy_final_hw)
print("hardware final expectations:", exp_final_hw)
print("hardware history:", energy_history_hw)


================== hardware full GD loop ==================
Baihua configuration loading done!
The last calibration time was 2026-03-06 13:40:51
iter=00 energy=0.683698 grad_norm=2.019686
iter=01 energy=0.405843 grad_norm=2.058671
iter=02 energy=-0.075503 grad_norm=2.150853
iter=03 energy=-0.633095 grad_norm=1.886150
iter=04 energy=-0.933668 grad_norm=1.684926
iter=05 energy=-1.043078 grad_norm=1.278695
hardware final energy: -1.131852376311925
hardware final expectations: {'Z0 Z1': 0.7891070195751102, 'X0': 0.09188296362063714, 'X1': 0.25086239311617775}
hardware history: [0.6836977305352102, 0.4058431888218927, -0.07550318338450418, -0.6330949524408274, -0.9336676830968343, -1.0430783446169027]


## 8) 下一步建议

- 把 `num_qubits` 增加到 4 或 6，观察参数量和收敛速度。
- 把手动 GD 改成 Adam（对应库内默认优化器）。